<h1><span style="color:red">SuAVE Transfer Learning Model</span></h1>

<h2> WORK IN PROGRESS </h2>
Author: Shuyuan Wang


## 1. Disable autoscroll and retrieve survey parameters from the URL

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: enter credentials if Drive had no session ----------------------
if su.ENV.colab:
    import getpass as _gp
    if _p is not None:
        _survey = _p.get('survey', '?')
        display(HTML(
            f'<p style="font-size:12px;margin:3px 0">Session active for survey '
            f'<b>{_survey}</b>. Press Enter to keep it, or paste a new '
            f'SUAVE_TOKEN to switch surveys.</p>'))
    else:
        display(HTML('<p style="color:#e07000;font-size:12px">No session loaded. '
                     'Enter credentials from SuAVEDispatch.</p>'))
    _tok = _gp.getpass('SUAVE_TOKEN (Enter to keep current): ')
    if _tok.strip():
        _host = input('SUAVE_HOST (e.g. george.tirebiter.org): ')
        _p = su.load_params(token=_tok.strip(), host=_host.strip())
    elif _p is None:
        raise RuntimeError(
            'No session and no token provided. '
            'Run SuAVEDispatch first, or enter SUAVE_TOKEN above.')
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    raise RuntimeError('Parameters not loaded. Run the cell above to enter credentials.')
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


## 2. Import all packages (this might take a few seconds)

In [7]:
# Import widget functionality
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')  # "error", "ignore", "always", "default", "module" or "once"
# import the necessary packages
import keras
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import img_to_array
from keras.utils import to_categorical
from imutils import paths

from keras.utils import np_utils
from keras.models import Sequential
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dense
from keras.layers.core import Dropout
from keras import backend as K
# import local lenet.py file describing the LeNet implementation with RELU activation functions
#from lenet import LeNet

# More imports
import matplotlib.pyplot as plt
import numpy as np
import argparse
import random
import csv
import pandas as pd
import re
import cv2
import os

# import the necessary packages for SVM predictor
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import imutils


# copy from sample
import pandas as pd
import numpy as np
import os
import keras
import matplotlib.pyplot as plt
from keras.layers import Dense,GlobalAveragePooling2D
from keras.applications import ResNet50
from keras.preprocessing import image
from keras.applications.mobilenet import preprocess_input
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Model
from keras.optimizers import Adam

import requests



def printmd(string):
    display(Markdown(string))


## 3. Initializing the number of times the model will loop and its batch size for learning

In [8]:
from collections import OrderedDict

epochCount = OrderedDict()
epochCount['25 Iterations'] = 25
epochCount['50 Iterations'] = 50
epochCount['75 Iterations'] = 75

def f(epoch_count):
    return epoch_count

epochNum = interact(f, epoch_count=epochCount)


In [9]:
batchS = OrderedDict()
batchS['32 Batch Size'] = 32
batchS['64 Batch Size'] = 64
batchS['128 Batch Size'] = 128

def f(batch_size):
    return batch_size

batchNum = interact(f, batch_size=batchS)

In [10]:
# init the number of epochs to train for, init learning rate and batch size
EPOCHS = epochNum.widget.result
INIT_LR = 1e-3
BS = batchNum.widget.result
# init the image suffix, yset, and image list
suffix = '.png'
img_list = []
yset = []
# create labels list and 2 dicts for 2 way mapping
labels = []
# key = label value = number
label_yval = dict()
# key = number value = label
yval_label = dict()

## 4. Load the data and choose the column name to be predicted

In [11]:
# use csv file to grab images/labels
# read the csv file
file = open(absolutePath + csv_file, encoding="latin-1")
df = pd.read_csv(file)

#generate image path
localdzc = dzc_file.replace("https://maxim.ucsd.edu/dzgen/lib-staging-uploads","/lib-nfs/dzgen")
full_images_location = localdzc.replace("/content.dzc","/full_images/")


# Choose column of label for prediction
toPredict = list(df.columns.values)

pred_menu = OrderedDict()
for i in range(0, len(toPredict)):
    pred_menu[toPredict[i]] = toPredict[i]

def f(predictions_menu):
    return predictions_menu
# choose which label for predictions
out2 = interact(f, predictions_menu=pred_menu)

## 5. Select pixel resolution

In [19]:
a = widgets.IntSlider(value=60,min=20,max=300,step=10,description='Size, pixels:')
display(a)

IntSlider(value=60, description='Size, pixels:', max=300, min=20, step=10)

In [20]:
printmd("<h2><span style='color:red'>Verify model parameters:</span></h2>")
printmd("<b>Variable to predict:  " +out2.widget.result + "</b>")
printmd("<b>Pixel resolution:  " +str(a.value) + "</b>")
printmd("<b>Path to images:  " +full_images_location + "</b>")
printmd("<b>Number of epochs:  " + str(EPOCHS) + "</b>")
printmd("<b>Batch size:  " +str(BS) + "</b>")


## 6. Grab the images and configure them for predicting

### This might take a little while depending on the size of the dataset

In [21]:
im_dimension = a.value
# grab chosen column names
nameCol = df['#img']
predCol = df[out2.widget.result]

def matchImage(curr_image_array, image_list):
    for i in range(0, len(image_list)):
        if (np.array_equal(curr_image_array, image_list[i])):
            return i
    
labels = []
# add all fabric columns to the y set
for i in range (0,len(predCol)):
    labels.append(predCol[i])

# grab all unique labels
uni_labels = set(labels)
uni_labels = list(uni_labels)

# assign each label a dict key number
for i in range(0,len(uni_labels)):
    yval_label[i] = uni_labels[i]
    label_yval[uni_labels[i]] = i


yset = []    
# create list of keys associated with their labels
for i in range (0, len(labels)):
    yset.append(label_yval[labels[i]])

img_list = []
file_lst=[]

counter = 0
im_count = widgets.Label(value="0% images loaded")
display(im_count)
numfiles = len(nameCol)

# gather images from path created from file names in csv file
for i in range (0,len(nameCol)):
    base_filename = nameCol[i]

    fileName = os.path.join(full_images_location, base_filename + suffix)
    
    file_lst.append(fileName)
    
    response = requests.get(fileName)
    if response.status_code == 200:
        # Read the image data into a NumPy array
        image_data = np.frombuffer(response.content, np.uint8)
        
        # Decode the image using OpenCV
        im = cv2.imdecode(image_data, cv2.IMREAD_COLOR)
        
        # Resize the image
        im_dimension = 256  # Adjust the dimension as needed
        im = cv2.resize(im, (im_dimension, im_dimension))
        
        # Convert the image to a Keras-compatible format
        im = img_to_array(im)
        
        # Append the processed image to the list
        img_list.append(im)
    else:
        print(f"Failed to fetch image from URL: {url}")
    
    counter += 1
    im_count.value = str(int(counter / numfiles * 100)) + "% images loaded"

# Shuffle the data
p = np.random.permutation(len(yset))

test_train_dict = {}
test_train_list = []

# Relable for splitting sets
Y = []
X = []
for i in range(0,len(yset)):
    Y.append(yset[p[i]])
    X.append(file_lst[p[i]])
    
# split the test and training set 75:25
split = int(len(X)*(.75))


In [22]:
ndf=pd.DataFrame()
ndf['id']=file_lst
ndf['label']=list(map(str, yset))

In [23]:
datagen = ImageDataGenerator(rescale=1/255., validation_split=0.25)
 
train_generator = datagen.flow_from_dataframe(dataframe=ndf, 
                                             x_col='id',
                                             y_col='label',
                                             target_size=(224,224),
                                              subset='training',
                                                 color_mode='rgb',
                                                 batch_size=32,
                                                 class_mode='binary',
                                              
                                                 shuffle=True)
 
validation_generator = datagen.flow_from_dataframe(dataframe=ndf, 
                                             x_col='id',
                                             y_col='label',
                                             target_size=(224,224),
                                                subset='validation',
                                                 color_mode='rgb',
                                                 batch_size=32,
                                                 class_mode='binary',
                                                 shuffle=True)


In [24]:
base_model=ResNet50(weights='imagenet',include_top=False)

newOutput=base_model.output
newOutput=GlobalAveragePooling2D()(newOutput)
newOutput=Dense(1024,activation='relu')(newOutput) 
newOutput=Dense(1024,activation='relu')(newOutput) 
newOutput=Dense(512,activation='relu')(newOutput)

preds=Dense(2,activation='softmax')(newOutput)


model=Model(inputs=base_model.input,outputs=preds)

model.compile(loss='binary_crossentropy',optimizer = 'Adam',metrics=['accuracy'])

train_steps = train_generator.n//train_generator.batch_size
validation_steps = validation_generator.n//validation_generator.batch_size

#history = model.fit_generator(train_generator,steps_per_epoch=train_steps, epochs=5,
                              

In [25]:
history = model.fit_generator(train_generator,steps_per_epoch=train_steps, epochs=5)

In [26]:
base_model=ResNet50(weights='imagenet',include_top=False)

newOutput=base_model.output
newOutput=GlobalAveragePooling2D()(newOutput)
newOutput=Dense(1024,activation='relu')(newOutput) 
newOutput=Dense(1024,activation='relu')(newOutput) 
newOutput=Dense(512,activation='relu')(newOutput)

preds=Dense(2,activation='softmax')(newOutput)


model=Model(inputs=base_model.input,outputs=preds)

model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['accuracy'])


step_size_train=train_generator.n//train_generator.batch_size
model.fit_generator(generator=train_generator,
                   steps_per_epoch=step_size_train,
                   epochs=5)

## STOP HERE

In [ ]:
def append_ext(fn):
    return fn+".png"
traindf=pd.read_csv("./trainLabels.csv",dtype=str)
testdf=pd.read_csv("./sampleSubmission.csv",dtype=str)
traindf["id"]=traindf["id"].apply(append_ext)
testdf["id"]=testdf["id"].apply(append_ext)
datagen=ImageDataGenerator(rescale=1./255.,validation_split=0.25)

train_generator=datagen.flow_from_dataframe(
dataframe=traindf,
directory="./train/",
x_col="id",
y_col="label",
subset="training",
batch_size=32,
seed=42,
shuffle=True,
class_mode="categorical",
target_size=(32,32))

In [ ]:
# Shuffle the data
p = np.random.permutation(len(yset))

test_train_dict = {}
test_train_list = []

# Relable for splitting sets
Y = []
X = []
for i in range(0,len(yset)):
    Y.append(yset[p[i]])
    X.append(img_list[p[i]])
    
# split the test and training set 75:25
split = int(len(X)*(.75))

## Original DO NOT DELETE
#xtrain = X[:split]
#xtest = X[split:]


## Testing new 
xtrain = []
xtest = []
for i in range(0, len(X)):
    
    if (i < split):
        curr_image_index = matchImage(X[i], img_list)
        test_train_dict[nameCol[curr_image_index]] = "train"
        xtrain.append(X[i])
    else:
        curr_image_index = matchImage(X[i], img_list)
        test_train_dict[nameCol[curr_image_index]] = "test"
        xtest.append(X[i])

    
    
ytrain = Y[:split]
ytest = Y[split:]

for i in range(0, len(nameCol)):
    #print(i)
    #print(nameCol[i])
    test_train_list.append(test_train_dict[nameCol[i]])


# transform to arrays
trainX = np.array(xtrain, dtype="float")/255.0
testX = np.array(xtest, dtype ="float")/255.0

ytrain = np.array(ytrain)
ytest = np.array(ytest)

# parsed Y data containers
trainY = []
testY = []


# convert labels from int to vectors
trainY = np_utils.to_categorical(ytrain,len(uni_labels))
testY = np_utils.to_categorical(ytest,len(uni_labels))

# construct the image generator for data augmentation
aug = ImageDataGenerator(rotation_range=30, width_shift_range=0.1,
                        height_shift_range=0.1, shear_range=0.2, zoom_range=0.2,
                        horizontal_flip=True, fill_mode="nearest")


In [ ]:
im = Image.fromarray(np.uint8(cm.gist_earth(myarray)*255))

In [ ]:
from PIL import Image
from matplotlib import cm

In [ ]:
im = Image.fromarray(np.uint8(cm.gist_earth(myarray)*255))

In [ ]:
myarray=trainX[0][0]

In [ ]:
trainY[0]

In [ ]:
# folder_path is set earlier in the notebook


In [ ]:
# initialize the model

model = LeNet.build(width=im_dimension, height=im_dimension, depth=3, classes=len(uni_labels))
opt = Adam(lr=INIT_LR, decay=INIT_LR / EPOCHS)
model.compile(loss="categorical_crossentropy", optimizer=opt,
                metrics=["accuracy"])

printmd("<b><span style='color:red'>Images split into training and testing sets at 75:25</span></b>")


In [ ]:
# sample
#train_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)
#train_generator=train_datagen.flow_from_directory(folder_path,
                                                 #target_size=(224,224),
                                                 #color_mode='rgb',
                                                 #batch_size=32,
                                                 #class_mode='categorical',
                                                 #shuffle=True)
# opt = Adam(lr=INIT_LR, decay=INIT_LR / EPOCHS)
#model.compile(optimizer=opt,loss='categorical_crossentropy',metrics=['accuracy'])



## 7. Train the predictive model

### This is relative to the size of the data set and the compute resources you use. May take from minutes to hours

In [ ]:
# sample
# step_size_train=train_generator.n//train_generator.batch_size
# model.fit_generator(generator=train_generator,
                   #steps_per_epoch=step_size_train,
                   #epochs=5)

In [ ]:
# train the network
H = model.fit_generator(aug.flow(trainX, trainY, batch_size=BS),
    validation_data=(testX, testY), steps_per_epoch=len(trainX) // BS,
    epochs=EPOCHS, verbose=1)
printmd("<b><span style='color:red'>Model generation complete</span></b>")

## 8. Take the original data and predict based on the new model

In [ ]:
# Reshape original input data images for predicting
img_check = np.array(img_list, dtype ="float")/255.0

predictionsMade = []
preds = model.predict(img_check)
prediction_confidence = []
for i in range(0, len(preds)):
    prediction_confidence.append(np.amax(preds[i]))   



# Run all data through the prediction model that was created
for i in range (0,len(img_check)):
    predIndex = np.where(preds[i] == np.amax(preds[i]))
    prediction = int(predIndex[0][0])
    predictionsMade.append(prediction)

print(prediction_confidence)    
# Count how many correct predictions were made
correct = 0
for i in range (0,len(predictionsMade)):
    if(predictionsMade[i] == yset[i]):
        correct += 1 
        
printmd("<b><span style='color:red'>Accuracy: " + str(correct/len(yset)) + "</span></b>")


## 9. Save the model file under models/, and reload it

In [ ]:
#Generate model file and save
modelName = user + "_cnn_" + out2.widget.result + "_" + str(epochNum.widget.result) + "_" + str(batchNum.widget.result) + ".h5"
modelPath = "models/"
if not os.path.exists(modelPath):
    os.makedirs(modelPath)
model.save(os.path.join(modelPath, modelName))
#Load model
from keras.models import load_model
model2 = load_model(os.path.join(modelPath, modelName))
printmd("<b><span style='color:red'>Model saved</span></b>")

## 10. Enter a new header for the prediction column

In [ ]:
# Translate back to original csv label names
finalPred = []
for i in range (0,len(predictionsMade)):
    finalPred.append(yval_label[predictionsMade[i]])

from IPython.display import display
input_text = widgets.Text(
    value="predicted " + out2.widget.result,
    placeholder='Type label here',
    disabled=False
)
output_text = widgets.Text(
    value="predicted " + out2.widget.result,
    placeholder='New Header will be displayed here',
    disabled=False
)

def bind_input_to_output(sender):
    output_text.value = input_text.value

input_text.observe(bind_input_to_output)

print("Input new column Header Label: ")

display(input_text)
display(output_text)

## 11. Save the new version of CSV file, and give a name to new survey

In [ ]:
# Append the new column w/ it's new column name
pred_conf_col = "pred_conf " + input_text.value
test_train_col = "test_train " + input_text.value

df[input_text.value] = finalPred
df[pred_conf_col] = prediction_confidence
df[test_train_col] = test_train_list

print(input_text.value)

new_file = absolutePath + csv_file[:-4]+'_v1.csv'
printmd("<b><span style='color:red'>A new temporary file will be created at: </span></b>")
print(new_file)
df.to_csv(new_file, index=None)



In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)


## 12. Generate the survey and create survey URL

In [ ]:
referer = survey_url.split("/main")[0] +"/"
upload_url = referer + "uploadCSV"
new_survey_url_base = survey_url.split(user)[0]

import requests
import re
csv = {"file": open(new_file, "rb")}
upload_data = {
    'name': input_text.value,
    'dzc': dzc_file,
    'user':user
}
headers = {
    'User-Agent': 'suave user agent',
    'referer': referer
}

r = requests.post(upload_url, files=csv, data=upload_data, headers=headers)

if r.status_code == 200:
    printmd("<b><span style='color:red'>New survey created successfully</span></b>")
    regex = re.compile('[^0-9a-zA-Z_]')
    s_url = survey_name
    s_url =  regex.sub('_', s_url)

    url = new_survey_url_base + user + "_" + s_url + ".csv" + "&views=" + views + "&view=" + view
    print(url)
    printmd("<b><span style='color:red'>Click the URL to open the new survey</span></b>")
else:
    printmd("<b><span style='color:red'>Error creating new survey. Check if a survey with this name already exists.</span></b>")
    printmd("<b><span style='color:red'>Reason: </span></b>"+ str(r.status_code) + " " + r.reason)

